### Which products have the highest revenue?

In [0]:
SELECT
product_id,
product_name,
SUM(revenue) AS product_revenue,
SUM(quantity) AS units_sold
FROM gold.fct_order_items
GROUP BY product_id,product_name
ORDER BY product_revenue DESC
LIMIT 20;

### Which categories drive the most revenue?

In [0]:
SELECT
    product_category AS category,
    SUM(revenue)     AS category_revenue,
    SUM(quantity)    AS units_sold
FROM gold.fct_order_items
GROUP BY product_category
ORDER BY category_revenue DESC;


###What is average order value (AOV) by month?

In [0]:
SELECT
    DATE_TRUNC('month', order_date)        AS month,
    SUM(revenue)                           AS total_revenue,
    COUNT(DISTINCT order_id)               AS order_count,
    SUM(revenue) / COUNT(DISTINCT order_id) AS aov
FROM gold.fct_orders
GROUP BY 1
ORDER BY 1;


###  Which countries generate the most revenue?

In [0]:
SELECT
    customer_country            AS country,
    SUM(revenue)                AS revenue,
    COUNT(DISTINCT order_id)    AS order_count
FROM gold.fct_orders
GROUP BY customer_country
ORDER BY revenue DESC
LIMIT 10;


### Who are the top customers by revenue?

In [0]:
SELECT
    o.customer_id,
    MAX(c.email)        AS email,
    MAX(c.country)      AS country,
    SUM(o.revenue)      AS total_revenue,
    COUNT(o.order_id)   AS order_count
FROM gold.fct_orders o
LEFT JOIN gold.scd_customers c
    ON o.customer_id = c.customer_id
   AND c.dbt_valid_to IS NULL
GROUP BY o.customer_id
ORDER BY total_revenue DESC
LIMIT 20;


### Who are the customers who have more than one order (repeat customers)?

In [0]:
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.no_of_orders
FROM (
    SELECT
        customer_id,
        COUNT(order_id) as no_of_orders
    FROM workspace.gold.fct_orders
    WHERE customer_id IS NOT NULL
      AND order_id IS NOT NULL
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
) o
JOIN workspace.gold.scd_customers c
    ON o.customer_id = c.customer_id;
